# Module 9 Instructor Colab Notebook: Random Variables and Probability Distributions

## A Mathematical Bridge to Artificial Intelligence

AI systems frequently represent uncertainty with probability distributions. A classifier may output the probability of a binary label, a count model may describe how many events occur, and parameter initialization may draw values from a Gaussian distribution. To interpret these operations correctly, students need to distinguish outcomes, random variables, probability masses, probability densities, cumulative probabilities, and distribution parameters.

This notebook develops Bernoulli, Binomial, Poisson, Gaussian, and Exponential distributions from their modeling assumptions. It then connects them to classification output layers, likelihood-based loss functions, random initialization, and simulation.

**Recommended pacing:** three class meetings. The mean and variance derivations may be shortened without interrupting the distribution-selection storyline.


> **How to use this notebook in class**
>
> Before naming a distribution, ask what values the random variable can take and what mechanism generates them. Keep PMF, PDF, and CDF units distinct. In the AI examples, emphasize that a model score becomes probabilistic only after a distributional interpretation is specified.

## Notebook map

| Section | Main mathematical ideas | Direct AI connection |
|---|---|---|
| 1 | Random variables, PMFs, PDFs, and CDFs | Representing uncertain model quantities |
| 2 | Bernoulli and Binomial distributions | Binary labels and batches of classifications |
| 3 | Poisson distribution | Counts of events or arrivals |
| 4 | Exponential and Gaussian distributions | Waiting times, noise, and continuous measurements |
| 5 | Sampling, mean, variance, and initialization | Stable random parameter scales |
| 6 | Likelihood and classification output layers | Cross-entropy training objectives |
| Integrated lab | Fit a Bernoulli model | Estimate and use a binary output probability |


In [ ]:
# Core setup: compatible with Google Colab and standard Jupyter.
import math
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

np.set_printoptions(precision=5, suppress=True)
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

try:
    import ipywidgets as widgets
    from ipywidgets import interact, FloatSlider, IntSlider
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False

print("Setup complete.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)


In [ ]:
# Reusable instructor-solution helper.
def reveal_markdown(button_text, solution_markdown):
    if not WIDGETS_AVAILABLE:
        display(Markdown("**Instructor solution**"))
        display(Markdown(solution_markdown))
        return

    button = widgets.Button(description=button_text, button_style="info")
    output = widgets.Output()
    state = {"open": False}

    def toggle(_):
        with output:
            output.clear_output()
            state["open"] = not state["open"]
            if state["open"]:
                display(Markdown(solution_markdown))
                button.description = "Hide instructor solution"
            else:
                button.description = button_text

    button.on_click(toggle)
    display(button, output)


## Module learning objectives

By the end of the module, students should be able to:

1. distinguish outcomes, events, and random variables;
2. distinguish discrete and continuous random variables;
3. compute probabilities from a PMF, PDF, or CDF;
4. identify the support and parameters of a distribution;
5. use Bernoulli, Binomial, Poisson, Exponential, and Gaussian models appropriately;
6. compute basic means and variances;
7. explain how variance affects random initialization; and
8. connect Bernoulli likelihood to binary cross-entropy.

## Prerequisite checkpoint

Students should know basic set notation, summation, factorials, exponentials, logarithms, integrals, and expected values.


---

# Section 1: Probability Language and Random Variables

## 1. Outcomes, sample spaces, and events

An **outcome** is one possible result of a random experiment. The **sample space** $$\Omega$$ is the set of all possible outcomes. An **event** is a subset of the sample space.

For one binary classification label,

$$
\Omega=\{0,1\}.
$$

The event that the label is positive is

$$
A=\{1\}.
$$

For two binary labels, the sample space is

$$
\Omega=\{(0,0),(0,1),(1,0),(1,1)\}.
$$

Probability assigns a number from zero to one to an event.


## 2. Random variables

A **random variable** is a function that assigns a numerical value to every outcome:

$$
X:\Omega\to\mathbb{R}.
$$

The randomness comes from the outcome; the mapping rule itself is fixed.

A **discrete random variable** takes a finite or countably infinite set of values, such as

$$
0,1,2,3,\ldots.
$$

A **continuous random variable** is modeled over intervals of real values, such as response time or measurement noise.

The **support** is the collection of values that the random variable can take with positive mass or density.


## 3. Probability mass functions

For a discrete random variable, the **probability mass function**, or **PMF**, is

$$
p_X(x)=P(X=x).
$$

It must satisfy

$$
p_X(x)\geq 0
$$

and

$$
\sum_x p_X(x)=1.
$$

For an event described by several possible values,

$$
P(X\in A)=\sum_{x\in A}p_X(x).
$$

A PMF value is a probability, unlike a continuous PDF value, which is a density height.


## 4. Probability density functions

For a continuous random variable, a **probability density function**, or **PDF**, satisfies

$$
f_X(x)\geq 0
$$

and

$$
\int_{-\infty}^{\infty}f_X(x)\,dx=1.
$$

Probability is obtained by integration:

$$
P(a\leq X\leq b)=\int_a^b f_X(x)\,dx.
$$

For a continuous model,

$$
P(X=x)=0
$$

for every individual point. Density measures local concentration per unit of the horizontal variable.


## 5. Cumulative distribution functions

For either a discrete or continuous random variable, the **cumulative distribution function**, or **CDF**, is

$$
F_X(x)=P(X\leq x).
$$

For a discrete random variable,

$$
F_X(x)=\sum_{t\leq x}p_X(t).
$$

For a continuous random variable,

$$
F_X(x)=\int_{-\infty}^{x}f_X(t)\,dt.
$$

Every CDF is nondecreasing, stays between zero and one, approaches zero in the far left tail, and approaches one in the far right tail.


In [ ]:
# A simple discrete distribution for the number of model alerts in one minute.
support_discrete9 = np.array([0, 1, 2, 3])
pmf_discrete9 = np.array([0.25, 0.40, 0.25, 0.10])
cdf_discrete9 = np.cumsum(pmf_discrete9)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].bar(support_discrete9, pmf_discrete9, width=0.65)
axes[0].set_xticks(support_discrete9)
axes[0].set_xlabel("number of alerts")
axes[0].set_ylabel("probability mass")
axes[0].set_title("PMF: probability at each value")
axes[1].step(support_discrete9, cdf_discrete9, where="post", linewidth=2)
axes[1].scatter(support_discrete9, cdf_discrete9, s=45)
axes[1].set_ylim(0, 1.05)
axes[1].set_xticks(support_discrete9)
axes[1].set_xlabel("threshold")
axes[1].set_ylabel("cumulative probability")
axes[1].set_title("CDF: probability at or below threshold")
plt.tight_layout()
plt.show()


## 6. Distribution parameters

A **probability distribution** specifies how probability is assigned across possible random-variable values. A **distribution parameter** controls its shape or location.

Examples include:

- Bernoulli success probability $$p$$;
- Binomial trial count $$n$$ and success probability $$p$$;
- Poisson rate $$\lambda$$;
- Gaussian mean $$\mu$$ and standard deviation $$\sigma$$; and
- Exponential rate $$\lambda.$$

A parameter of a probability model should not be confused automatically with a trainable neural-network weight, although a network can output or learn distribution parameters.


> **Common mistakes in Section 1**
>
> - Confusing an outcome with a random-variable value.
> - Calling a continuous PDF value a probability.
> - Forgetting that total mass or area must equal one.
> - Treating a CDF as though it can decrease.
> - Confusing a distribution's support with one observed sample.
> - Assuming every parameter is a neural-network weight.

## Section 1 terminology

| Term | Meaning |
|---|---|
| Outcome | One possible result of a random experiment |
| Sample space | Set of all possible outcomes |
| Event | Subset of the sample space |
| Random variable | Numerical function of a random outcome |
| Discrete random variable | Variable with finite or countable support |
| Continuous random variable | Variable modeled across real intervals |
| PMF | Probability assigned to each discrete value |
| PDF | Density whose interval integrals give probabilities |
| CDF | Probability accumulated up to a threshold |
| Support | Values where a distribution places mass or density |


## Section 1 guided worksheet

1. Classify each random variable as discrete or continuous:

   - number of incorrect predictions in a batch;
   - inference latency in seconds;
   - predicted class label;
   - sensor voltage.

2. A PMF assigns masses $$0.2,$$ $$0.5,$$ and $$c$$ to values zero, one, and two. Find $$c.$$
3. Compute the CDF at one.
4. Explain why a PDF height of $$1.4$$ does not violate probability rules.


In [ ]:
reveal_markdown("Open Section 1 solutions", r"""
1. Counts and class labels are discrete. Latency and voltage are modeled as continuous.

2. Total probability must equal one:

   $$
   0.2+0.5+c=1.
   $$

   Therefore

   $$
   \boxed{c=0.3}.
   $$

3. The CDF at one includes masses at zero and one:

   $$
   F_X(1)=0.2+0.5=\boxed{0.7}.
   $$

4. A continuous density is probability per unit, not probability itself. Its height may exceed one when concentrated over a narrow interval, provided its total integral is one and it remains nonnegative.
""")


---

# Section 2: Bernoulli and Binomial Distributions

## 1. Bernoulli random variables

A **Bernoulli random variable** represents one trial with two encoded outcomes:

$$
X=
\begin{cases}
1,&\text{success},\\
0,&\text{failure}.
\end{cases}
$$

If

$$
P(X=1)=p,
$$

then

$$
P(X=0)=1-p,
\qquad 0\leq p\leq 1.
$$

Its PMF can be written compactly as

$$
\boxed{P(X=x)=p^x(1-p)^{1-x},
\qquad x\in\{0,1\}.}
$$

When $$x=1,$$ the formula becomes $$p.$$ When $$x=0,$$ it becomes $$1-p.$$


## 2. Mean and variance of a Bernoulli variable

The expected value is

$$
E[X]=0(1-p)+1(p)=\boxed{p}.
$$

Because $$X^2=X$$ for values zero and one,

$$
E[X^2]=E[X]=p.
$$

The variance is

$$
\operatorname{Var}(X)=E[X^2]-E[X]^2.
$$

Therefore

$$
\operatorname{Var}(X)=p-p^2
=\boxed{p(1-p)}.
$$

The variance is largest at $$p=1/2$$ and vanishes at $$p=0$$ or $$p=1.$$


## 3. Binomial random variables

A **Binomial random variable** counts successes in $$n$$ independent Bernoulli trials having the same success probability $$p.$$

If

$$
X\sim\operatorname{Binomial}(n,p),
$$

then

$$
\boxed{P(X=k)=\binom{n}{k}p^k(1-p)^{n-k},
\qquad k=0,1,\ldots,n.}
$$

The binomial coefficient

$$
\binom{n}{k}=\frac{n!}{k!(n-k)!}
$$

counts which $$k$$ of the $$n$$ trials are successes. Every such arrangement has probability

$$
p^k(1-p)^{n-k}.
$$


## 4. Detailed Binomial calculation

Suppose five independent classifications each have probability $$0.3$$ of being incorrect. Let $$X$$ count incorrect predictions. Then

$$
X\sim\operatorname{Binomial}(5,0.3).
$$

The probability of exactly two errors is

$$
P(X=2)=\binom{5}{2}(0.3)^2(0.7)^3.
$$

Compute the combinatorial factor:

$$
\binom{5}{2}=\frac{5!}{2!3!}=10.
$$

Compute the powers:

$$
(0.3)^2=0.09,
$$

$$
(0.7)^3=0.343.
$$

Therefore

$$
P(X=2)=10(0.09)(0.343)
=\boxed{0.3087}.
$$


In [ ]:
def binomial_pmf(k, n, p):
    return math.comb(n, k)*(p**k)*((1-p)**(n-k))

n_binomial9 = 10
p_values_binomial9 = [0.2, 0.5, 0.8]
k_binomial9 = np.arange(n_binomial9+1)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, p_value in zip(axes, p_values_binomial9):
    masses = np.array([binomial_pmf(int(k), n_binomial9, p_value) for k in k_binomial9])
    ax.bar(k_binomial9, masses)
    ax.set_xticks(k_binomial9)
    ax.set_xlabel("success count")
    ax.set_ylabel("probability")
    ax.set_title(f"n = 10, p = {p_value}")
plt.tight_layout()
plt.show()


## 5. Binomial mean and variance

Write a Binomial count as a sum of independent Bernoulli indicators:

$$
X=X_1+X_2+\cdots+X_n.
$$

By linearity of expectation,

$$
E[X]=E[X_1]+\cdots+E[X_n]
=np.
$$

Independence allows variances to add:

$$
\operatorname{Var}(X)
=\operatorname{Var}(X_1)+\cdots+\operatorname{Var}(X_n)
=np(1-p).
$$

Therefore

$$
\boxed{E[X]=np,
\qquad
\operatorname{Var}(X)=np(1-p).}
$$


## 6. Binary classification interpretation

For a binary target

$$
Y\in\{0,1\},
$$

a classifier may output

$$
p=P(Y=1\mid\mathbf{x}).
$$

The sigmoid converts a real score, or **logit**, into a valid probability:

$$
p=\sigma(z)=\frac{1}{1+e^{-z}}.
$$

The Bernoulli model then assigns

$$
P(Y=y\mid\mathbf{x})=p^y(1-p)^{1-y}.
$$

The model does not claim the label itself is uncertain after observation. It represents uncertainty before the label is known, conditional on the available features.


> **Common mistakes in Section 2**
>
> - Using Bernoulli for a count larger than one.
> - Using Binomial without a fixed trial count.
> - Omitting the combinatorial coefficient.
> - Treating dependent trials as automatically Binomial.
> - Confusing the logit with the probability.
> - Interpreting a predicted probability as a guaranteed frequency in a tiny sample.

## Section 2 terminology

| Term | Meaning |
|---|---|
| Bernoulli distribution | Distribution of one zero-or-one trial |
| Success probability | Parameter $$p$$ assigned to value one |
| Binomial distribution | Count of successes in fixed independent Bernoulli trials |
| Binomial coefficient | Number of ways to position a selected number of successes |
| Indicator variable | Zero-or-one variable marking whether an event occurs |
| Logit | Real-valued score converted to probability by sigmoid |
| Binary classifier | Model predicting between two label categories |


## Section 2 guided worksheet

Suppose eight independently selected images each have probability $$0.25$$ of requiring manual review. Let $$X$$ be the review count.

1. Identify the distribution and its parameters.
2. Compute $$P(X=0).$$
3. Compute $$P(X=1).$$
4. Compute the mean and variance.
5. State which assumption would fail if images arrived in highly similar duplicate groups.


In [ ]:
reveal_markdown("Open Section 2 solutions", r"""
1. We have

   $$
   X\sim\operatorname{Binomial}(8,0.25).
   $$

2. The probability of no reviews is

   $$
   P(X=0)=\binom{8}{0}(0.25)^0(0.75)^8
   =(0.75)^8
   \approx \boxed{0.100113}.
   $$

3. The probability of exactly one is

   $$
   P(X=1)=\binom{8}{1}(0.25)(0.75)^7
   =8(0.25)(0.75)^7
   \approx \boxed{0.266968}.
   $$

4. The mean is

   $$
   E[X]=np=8(0.25)=\boxed{2}.
   $$

   The variance is

   $$
   \operatorname{Var}(X)=np(1-p)
   =8(0.25)(0.75)
   =\boxed{1.5}.
   $$

5. Duplicate groups can make review outcomes dependent, violating the independent-trial assumption.
""")


---

# Section 3: The Poisson Distribution

## 1. Modeling event counts

A **Poisson random variable** models a count of events in a fixed interval of time, space, area, or volume when events occur with a stable average rate and approximately independent increments.

If

$$
X\sim\operatorname{Poisson}(\lambda),
$$

then

$$
\boxed{P(X=k)=e^{-\lambda}\frac{\lambda^k}{k!},
\qquad k=0,1,2,\ldots.}
$$

The parameter $$\lambda>0$$ is the expected event count in the chosen interval. Changing the interval usually changes $$\lambda.$$


## 2. Why the PMF sums to one

Sum over all nonnegative counts:

$$
\sum_{k=0}^{\infty}e^{-\lambda}\frac{\lambda^k}{k!}
=e^{-\lambda}\sum_{k=0}^{\infty}\frac{\lambda^k}{k!}.
$$

Using the exponential series

$$
e^{\lambda}=\sum_{k=0}^{\infty}\frac{\lambda^k}{k!},
$$

we obtain

$$
e^{-\lambda}e^{\lambda}=1.
$$

Thus the Poisson PMF is normalized.


## 3. Detailed cumulative probability

Suppose an AI service receives an average of three unusual requests per minute. Model

$$
X\sim\operatorname{Poisson}(3).
$$

The probability of at most two unusual requests is

$$
P(X\leq 2)=P(X=0)+P(X=1)+P(X=2).
$$

Substitute the PMF:

$$
P(X\leq 2)
=e^{-3}\frac{3^0}{0!}
+e^{-3}\frac{3^1}{1!}
+e^{-3}\frac{3^2}{2!}.
$$

Factor $$e^{-3}:$$

$$
P(X\leq 2)
=e^{-3}\left(1+3+\frac{9}{2}\right).
$$

Since

$$
1+3+\frac{9}{2}=8.5,
$$

we obtain

$$
\boxed{P(X\leq 2)=8.5e^{-3}\approx 0.423190.}
$$


## 4. Poisson mean and variance

For a Poisson random variable,

$$
\boxed{E[X]=\lambda}
$$

and

$$
\boxed{\operatorname{Var}(X)=\lambda.}
$$

This equality of mean and variance is a distinctive property. Real count data often show larger variance than their mean, called **overdispersion**, or smaller variance, called **underdispersion**. In such cases a basic Poisson model may be inadequate.


In [ ]:
def poisson_pmf(k, rate):
    return np.exp(-rate)*(rate**k)/math.factorial(k)

rates_poisson9 = [1.0, 3.0, 7.0]
k_poisson9 = np.arange(0, 18)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, rate in zip(axes, rates_poisson9):
    masses = np.array([poisson_pmf(int(k), rate) for k in k_poisson9])
    ax.bar(k_poisson9, masses)
    ax.set_xlabel("event count")
    ax.set_ylabel("probability")
    ax.set_title(f"Poisson rate = {rate}")
plt.tight_layout()
plt.show()


## 5. Poisson approximation to Binomial counts

When $$n$$ is large, $$p$$ is small, and

$$
np=\lambda
$$

is moderate, a Binomial distribution can be approximated by a Poisson distribution with rate $$\lambda.$$

This is useful for rare events among many opportunities. It is an approximation, not an identity. Dependence, changing probabilities, or event clustering can invalidate the model.


> **Common mistakes in Section 3**
>
> - Using Poisson for continuous measurements instead of counts.
> - Forgetting that the rate depends on interval length.
> - Treating equality of mean and variance as universal for count data.
> - Applying a Poisson model to strongly clustered or dependent events without checking.
> - Confusing the Poisson rate with an event probability.

## Section 3 terminology

| Term | Meaning |
|---|---|
| Poisson distribution | Distribution for nonnegative event counts |
| Rate parameter | Expected event count in the chosen interval |
| Independent increments | Counts in disjoint intervals are independent under the model |
| Overdispersion | Variance larger than the mean |
| Underdispersion | Variance smaller than the mean |
| Rare-event approximation | Poisson approximation to a large-trial, small-probability Binomial model |


## Section 3 guided worksheet

Suppose model failures occur according to a Poisson distribution with average rate $$1.5$$ per day.

1. Compute the probability of zero failures tomorrow.
2. Compute the probability of exactly one failure.
3. Compute the probability of at least one failure.
4. State the model's mean and variance.
5. Explain why a two-day interval would use a different rate.


In [ ]:
reveal_markdown("Open Section 3 solutions", r"""
1. For $$X\sim\operatorname{Poisson}(1.5),$$

   $$
   P(X=0)=e^{-1.5}\frac{1.5^0}{0!}
   =e^{-1.5}
   \approx \boxed{0.223130}.
   $$

2. We have

   $$
   P(X=1)=e^{-1.5}\frac{1.5}{1}
   \approx \boxed{0.334695}.
   $$

3. Use the complement:

   $$
   P(X\geq 1)=1-P(X=0)
   =1-e^{-1.5}
   \approx \boxed{0.776870}.
   $$

4. Both the mean and variance equal $$\lambda=1.5.$$

5. If the rate is stable, doubling the observation time doubles the expected count, so the two-day rate would be $$3.$$
""")


---

# Section 4: Exponential and Gaussian Distributions

## 1. Exponential waiting times

The Exponential distribution models a nonnegative waiting time under a constant event rate. Its PDF is

$$
f_X(x)=
\begin{cases}
\lambda e^{-\lambda x},&x\geq 0,\\
0,&x<0,
\end{cases}
$$

with rate $$\lambda>0.$$ Its CDF is

$$
F_X(x)=
\begin{cases}
0,&x<0,\\
1-e^{-\lambda x},&x\geq 0.
\end{cases}
$$

The survival probability is

$$
P(X>x)=e^{-\lambda x}.
$$


## 2. Detailed Exponential probability

Suppose an inference request has an idealized Exponential waiting-time model with rate

$$
\lambda=2
$$

per second. The probability that the waiting time exceeds one second is

$$
P(X>1)=e^{-2(1)}=\boxed{e^{-2}\approx 0.135335}.
$$

The probability that it lies between one-half and one second is

$$
P(0.5\leq X\leq 1)
=F_X(1)-F_X(0.5).
$$

Substitute the CDF:

$$
P=(1-e^{-2})-(1-e^{-1})
=e^{-1}-e^{-2}
\approx \boxed{0.232544}.
$$


## 3. Gaussian distribution

A Gaussian random variable with mean $$\mu$$ and variance $$\sigma^2$$ has PDF

$$
\boxed{f_X(x)=\frac{1}{\sigma\sqrt{2\pi}}
\exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right),
\qquad \sigma>0.}
$$

We write

$$
X\sim\mathcal{N}(\mu,\sigma^2).
$$

The mean $$\mu$$ controls location. The standard deviation $$\sigma$$ controls spread. The variance is $$\sigma^2.$$

The **standard normal** has

$$
\mu=0
\qquad\text{and}\qquad
\sigma=1.
$$


## 4. Standardization

If

$$
X\sim\mathcal{N}(\mu,\sigma^2),
$$

then the standardized variable

$$
Z=\frac{X-\mu}{\sigma}
$$

has a standard normal distribution.

For example, if

$$
X\sim\mathcal{N}(10,4),
$$

then $$\sigma=2.$$ The value $$X=13$$ has standardized score

$$
z=\frac{13-10}{2}=\boxed{1.5}.
$$

This says the value is one and one-half standard deviations above the mean.


In [ ]:
x_continuous9 = np.linspace(-5, 7, 700)

def gaussian_pdf(x, mean, standard_deviation):
    return np.exp(-0.5*((x-mean)/standard_deviation)**2)/(standard_deviation*np.sqrt(2*np.pi))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for mean, standard_deviation in [(0, 0.7), (0, 1.5), (2, 1.0)]:
    axes[0].plot(x_continuous9, gaussian_pdf(x_continuous9, mean, standard_deviation),
                 label=f"mean = {mean}, sd = {standard_deviation}")
axes[0].set_xlabel("x")
axes[0].set_ylabel("density")
axes[0].set_title("Gaussian location and spread")
axes[0].legend(fontsize=8)
x_exp9 = np.linspace(0, 5, 500)
for rate in [0.5, 1.0, 2.0]:
    axes[1].plot(x_exp9, rate*np.exp(-rate*x_exp9), label=f"rate = {rate}")
axes[1].set_xlabel("x")
axes[1].set_ylabel("density")
axes[1].set_title("Exponential rate controls tail scale")
axes[1].legend()
plt.tight_layout()
plt.show()


## 5. Memoryless property of the Exponential distribution

The Exponential distribution satisfies

$$
P(X>s+t\mid X>s)=P(X>t).
$$

Using conditional probability,

$$
P(X>s+t\mid X>s)
=\frac{P(X>s+t)}{P(X>s)}.
$$

Substitute the survival function:

$$
\frac{e^{-\lambda(s+t)}}{e^{-\lambda s}}
=e^{-\lambda t}.
$$

Since

$$
e^{-\lambda t}=P(X>t),
$$

the remaining waiting-time distribution does not depend on how long has already elapsed. This property is mathematically convenient but not realistic for every timing process.


> **Common mistakes in Section 4**
>
> - Using an Exponential model for values that can be negative.
> - Confusing the Exponential rate with its mean; the mean is the reciprocal rate.
> - Writing a Gaussian standard deviation where the variance parameter is expected.
> - Interpreting a standardized score as a probability.
> - Assuming all bell-shaped data are exactly Gaussian.
> - Applying memorylessness to arbitrary waiting-time distributions.

## Section 4 terminology

| Term | Meaning |
|---|---|
| Exponential distribution | Continuous distribution for nonnegative waiting times |
| Survival function | Probability of exceeding a threshold |
| Memoryless property | Remaining wait distribution independent of elapsed wait |
| Gaussian distribution | Symmetric bell-shaped continuous distribution |
| Mean parameter | Center of a Gaussian distribution |
| Variance parameter | Squared spread of a Gaussian distribution |
| Standardization | Subtracting the mean and dividing by standard deviation |
| Standard score | Standardized distance from the mean |


## Section 4 guided worksheet

1. If $$X$$ is Exponential with rate $$0.5,$$ find $$P(X>4).$$
2. Find its mean.
3. If $$Y\sim\mathcal{N}(20,9),$$ identify the mean and standard deviation.
4. Standardize the value $$Y=14.$$
5. Explain why an Exponential distribution is right-skewed while a Gaussian distribution is symmetric.


In [ ]:
reveal_markdown("Open Section 4 solutions", r"""
1. The survival probability is

   $$
   P(X>4)=e^{-0.5(4)}=\boxed{e^{-2}\approx 0.135335}.
   $$

2. The Exponential mean is

   $$
   E[X]=\frac{1}{\lambda}=\frac{1}{0.5}=\boxed{2}.
   $$

3. The notation gives

   $$
   \mu=20
   $$

   and

   $$
   \sigma^2=9.
   $$

   Therefore

   $$
   \sigma=3.
   $$

4. The standardized score is

   $$
   z=\frac{14-20}{3}=\frac{-6}{3}=\boxed{-2}.
   $$

5. Exponential support begins at zero and its density decreases in one direction, producing a long right tail. A Gaussian density depends on the squared distance from its mean, so equal distances on either side have equal density.
""")


---

# Section 5: Sampling, Mean, Variance, and AI Initialization

## 1. Population quantities and sample quantities

A distribution describes a population model. A **random sample** consists of repeated draws

$$
X_1,X_2,\ldots,X_n.
$$

The sample mean is

$$
\overline{X}_n=\frac{1}{n}\sum_{i=1}^{n}X_i.
$$

The **law of large numbers** says that under standard conditions, the sample mean approaches the population mean as sample size grows:

$$
\overline{X}_n\to E[X].
$$

This does not mean every finite sample is close, nor does it guarantee monotone improvement as samples are added.


## 2. Variance and signal scale

Variance measures average squared distance from the mean:

$$
\operatorname{Var}(X)=E[(X-E[X])^2].
$$

The standard deviation is

$$
\operatorname{SD}(X)=\sqrt{\operatorname{Var}(X)}.
$$

For independent random variables,

$$
\operatorname{Var}\left(\sum_i a_iX_i\right)
=\sum_i a_i^2\operatorname{Var}(X_i).
$$

This formula explains why the scale of random weights affects the scale of activations.


## 3. A simplified initialization calculation

Consider a linear pre-activation

$$
Z=\sum_{i=1}^{d}W_iX_i.
$$

Assume the terms are independent, have zero means, and

$$
\operatorname{Var}(W_i)=\sigma_W^2,
\qquad
\operatorname{Var}(X_i)=\sigma_X^2.
$$

Then each product has approximate variance

$$
\operatorname{Var}(W_iX_i)=\sigma_W^2\sigma_X^2.
$$

Adding $$d$$ independent contributions gives

$$
\operatorname{Var}(Z)
=d\sigma_W^2\sigma_X^2.
$$

If one wants the output variance to remain roughly comparable to the input variance, choose the weight variance on the scale

$$
\sigma_W^2\approx\frac{1}{d}.
$$

The input count $$d$$ is often called **fan-in**. The **fan-out** is the number of units receiving outputs from the current layer. More specialized initialization rules also account for activation functions and fan-out.


In [ ]:
rng_initialization9 = np.random.default_rng(2027)
fan_in_values9 = [10, 100, 1000]
sample_count9 = 8000
variances_scaled9 = []
variances_unscaled9 = []

for fan_in in fan_in_values9:
    inputs = rng_initialization9.normal(0, 1, size=(sample_count9, fan_in))
    weights_scaled = rng_initialization9.normal(0, np.sqrt(1/fan_in), size=fan_in)
    weights_unscaled = rng_initialization9.normal(0, 1, size=fan_in)
    variances_scaled9.append(np.var(inputs @ weights_scaled))
    variances_unscaled9.append(np.var(inputs @ weights_unscaled))

fig, ax = plt.subplots()
x_positions9 = np.arange(len(fan_in_values9))
width9 = 0.36
ax.bar(x_positions9-width9/2, variances_scaled9, width9, label="weight variance = one over fan-in")
ax.bar(x_positions9+width9/2, variances_unscaled9, width9, label="weight variance = one")
ax.set_xticks(x_positions9, fan_in_values9)
ax.set_xlabel("fan-in")
ax.set_ylabel("empirical pre-activation variance")
ax.set_yscale("log")
ax.set_title("Initialization scale controls signal variance")
ax.legend()
plt.show()


## 4. Sampling visualizations

A histogram summarizes a finite sample, while a PDF describes a continuous population model. The histogram depends on random observations and bin choices. The PDF does not.

As sample size grows, a properly scaled histogram often resembles the generating density more closely, but finite-sample fluctuations remain.


In [ ]:
rng_sampling9 = np.random.default_rng(77)
sizes_sampling9 = [40, 400, 4000]
x_density_sampling9 = np.linspace(-4, 4, 500)
standard_density_sampling9 = gaussian_pdf(x_density_sampling9, 0, 1)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, size in zip(axes, sizes_sampling9):
    sample = rng_sampling9.normal(0, 1, size=size)
    ax.hist(sample, bins=20, density=True, alpha=0.45, label="sample histogram")
    ax.plot(x_density_sampling9, standard_density_sampling9, linewidth=2,
            label="population PDF")
    ax.set_title(f"sample size = {size}")
    ax.set_xlabel("value")
    ax.set_ylabel("density")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()


> **Common mistakes in Section 5**
>
> - Treating a histogram as the exact population distribution.
> - Confusing variance with standard deviation.
> - Forgetting squared coefficients in the variance of a linear combination.
> - Assuming the law of large numbers makes every finite sample accurate.
> - Choosing initialization scale without considering fan-in or activation behavior.
> - Believing zero-mean initialization means every sampled weight is zero.

## Section 5 terminology

| Term | Meaning |
|---|---|
| Random sample | Collection of repeated draws from a distribution |
| Sample mean | Arithmetic average of observed sample values |
| Law of large numbers | Convergence of sample averages to population expectation |
| Variance | Expected squared deviation from the mean |
| Standard deviation | Square root of variance |
| Initialization | Assignment of starting parameter values before training |
| Fan-in | Number of inputs entering a unit |
| Histogram | Binned summary of observed sample values |


## Section 5 guided worksheet

Suppose

$$
Z=\sum_{i=1}^{50}W_iX_i,
$$

where the independent terms have zero means,

$$
\operatorname{Var}(X_i)=1,
$$

and

$$
\operatorname{Var}(W_i)=0.02.
$$

1. Compute the approximate variance of one product.
2. Compute the approximate variance of $$Z.$$
3. Explain why $$0.02=1/50$$ is a meaningful scale.
4. Explain why a new random initialization need not produce exactly the same empirical variance.


In [ ]:
reveal_markdown("Open Section 5 solutions", r"""
1. Under the stated independence and zero-mean assumptions,

   $$
   \operatorname{Var}(W_iX_i)
   =\operatorname{Var}(W_i)\operatorname{Var}(X_i)
   =(0.02)(1)
   =\boxed{0.02}.
   $$

2. Add the fifty independent variances:

   $$
   \operatorname{Var}(Z)=50(0.02)=\boxed{1}.
   $$

3. The value is the reciprocal of fan-in, so summing fifty contributions approximately cancels the factor fifty and preserves unit variance.

4. Variance one is a population-scale target. A finite random draw has sampling variation, so its empirical variance changes across initializations.
""")


---

# Section 6: Likelihood and Classification Output Layers

## 1. Likelihood

A **likelihood** treats observed data as fixed and views their probability as a function of an unknown model parameter.

For independent observations $$x_1,\ldots,x_n$$ with PMF or PDF $$p(x;\theta),$$

$$
\mathcal{L}(\theta)
=\prod_{i=1}^{n}p(x_i;\theta).
$$

The **log-likelihood** is

$$
\ell(\theta)
=\log\mathcal{L}(\theta)
=\sum_{i=1}^{n}\log p(x_i;\theta).
$$

Logs convert products into sums and often improve numerical stability.


## 2. Bernoulli negative log-likelihood

For one observed binary label $$y$$ and predicted probability $$p,$$ the Bernoulli likelihood is

$$
p^y(1-p)^{1-y}.
$$

Take the logarithm:

$$
y\ln p+(1-y)\ln(1-p).
$$

Training minimizes the negative log-likelihood:

$$
\boxed{-\left[y\ln p+(1-y)\ln(1-p)\right].}
$$

This is the **binary cross-entropy loss**.

If $$y=1,$$ the loss is

$$
-\ln p,
$$

which becomes large when the model assigns a small probability to the observed positive label. If $$y=0,$$ the loss is

$$
-\ln(1-p).
$$


In [ ]:
p_crossentropy9 = np.linspace(0.001, 0.999, 500)
loss_positive9 = -np.log(p_crossentropy9)
loss_negative9 = -np.log(1-p_crossentropy9)

fig, ax = plt.subplots()
ax.plot(p_crossentropy9, loss_positive9, linewidth=2, label="observed label = 1")
ax.plot(p_crossentropy9, loss_negative9, linewidth=2, label="observed label = 0")
ax.set_xlabel("predicted probability of label one")
ax.set_ylabel("binary cross-entropy")
ax.set_ylim(0, 7)
ax.set_title("Confident wrong predictions receive large loss")
ax.legend()
plt.show()


## 3. Categorical output layer

For $$K$$ mutually exclusive classes, a **Categorical distribution** assigns probabilities

$$
p_1,p_2,\ldots,p_K,
$$

with

$$
p_k\geq 0
$$

and

$$
\sum_{k=1}^{K}p_k=1.
$$

Softmax converts scores $$z_1,\ldots,z_K$$ into these probabilities:

$$
p_k=\frac{e^{z_k}}{\sum_{j=1}^{K}e^{z_j}}.
$$

If the observed class is $$c,$$ the categorical negative log-likelihood is

$$
-\ln p_c.
$$

Bernoulli is appropriate for one binary label. Categorical is appropriate for one label chosen from more than two mutually exclusive classes.


## 4. Calibration versus classification

A **decision threshold** converts a probability into a class label, for example predicting one when

$$
p\geq 0.5.
$$

**Calibration** asks whether cases receiving probability near $$0.7$$ are positive about seventy percent of the time over comparable cases. **Accuracy** asks how often thresholded labels are correct.

Two models can have similar classification accuracy but different probability quality. Distributional outputs matter when decisions depend on risk, uncertainty, ranking, or cost.


> **Common mistakes in Section 6**
>
> - Confusing likelihood with the probability distribution of a parameter.
> - Multiplying many tiny probabilities without using logs.
> - Taking the logarithm of zero without numerical safeguards.
> - Using a Categorical distribution for independent multi-label outcomes.
> - Equating high accuracy with well-calibrated probabilities.
> - Treating the threshold as part of the Bernoulli distribution itself.

## Section 6 terminology

| Term | Meaning |
|---|---|
| Likelihood | Data probability viewed as a function of model parameters |
| Log-likelihood | Logarithm converting likelihood products to sums |
| Negative log-likelihood | Loss obtained by negating the log-likelihood |
| Binary cross-entropy | Bernoulli negative log-likelihood |
| Categorical distribution | Distribution over mutually exclusive class labels |
| Softmax | Transformation from real scores to class probabilities |
| Decision threshold | Rule converting probability into a predicted label |
| Calibration | Agreement between predicted probabilities and observed frequencies |


## Section 6 guided worksheet

1. Compute binary cross-entropy when $$y=1$$ and $$p=0.8.$$
2. Compute it when $$y=1$$ and $$p=0.1.$$
3. Explain which prediction is penalized more and why.
4. For three class scores $$0,$$ $$1,$$ and $$2,$$ write the softmax probability formula for the third class.
5. Explain why three independent yes-or-no labels are not one Categorical outcome.


In [ ]:
reveal_markdown("Open Section 6 solutions", r"""
1. For a positive observed label,

   $$
   -\ln(0.8)\approx \boxed{0.223144}.
   $$

2. The loss is

   $$
   -\ln(0.1)\approx \boxed{2.302585}.
   $$

3. Probability $$0.1$$ is penalized much more because the model assigned low probability to the label that actually occurred.

4. The third-class probability is

   $$
   \boxed{p_3=\frac{e^2}{e^0+e^1+e^2}}.
   $$

5. A Categorical variable chooses exactly one mutually exclusive class. Independent yes-or-no labels can have several positive labels simultaneously and require separate Bernoulli variables or another multi-label model.
""")


---

# Integrated Instructor Lab: Fit and Use a Bernoulli Output Model

Suppose observed binary labels are

$$
1,0,1,1,0.
$$

A constant model predicts the same positive probability $$p$$ for every case. There are three positive labels and two negative labels.

The likelihood is

$$
\mathcal{L}(p)=p^3(1-p)^2.
$$

The log-likelihood is

$$
\ell(p)=3\ln p+2\ln(1-p).
$$

Differentiate:

$$
\ell'(p)=\frac{3}{p}-\frac{2}{1-p}.
$$

Set the derivative to zero:

$$
\frac{3}{p}=\frac{2}{1-p}.
$$

Cross multiply:

$$
3(1-p)=2p.
$$

Expand:

$$
3-3p=2p.
$$

Therefore

$$
3=5p
$$

and

$$
\boxed{\widehat{p}=\frac{3}{5}=0.6.}
$$

The maximum-likelihood estimate is the observed positive fraction.


## Curvature and future-count probabilities

The second derivative is

$$
\ell''(p)=-\frac{3}{p^2}-\frac{2}{(1-p)^2}<0
$$

for $$0<p<1,$$ so the critical point maximizes log-likelihood. Equivalently, it minimizes negative log-likelihood.

If the next five labels are modeled as independent Bernoulli trials with $$p=0.6,$$ the number of positives is

$$
X\sim\operatorname{Binomial}(5,0.6).
$$

The probability of exactly four positives is

$$
P(X=4)=\binom{5}{4}(0.6)^4(0.4).
$$

Thus

$$
P(X=4)=5(0.1296)(0.4)=\boxed{0.2592}.
$$

The probability of at least one positive is

$$
P(X\geq 1)=1-P(X=0)
=1-(0.4)^5
=\boxed{0.98976}.
$$


In [ ]:
p_grid_lab9 = np.linspace(0.01, 0.99, 500)
negative_log_likelihood_lab9 = -(3*np.log(p_grid_lab9)+2*np.log(1-p_grid_lab9))
p_hat_lab9 = 3/5
nll_at_hat_lab9 = -(3*np.log(p_hat_lab9)+2*np.log(1-p_hat_lab9))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(p_grid_lab9, negative_log_likelihood_lab9, linewidth=2)
axes[0].scatter([p_hat_lab9], [nll_at_hat_lab9], color="crimson", s=80,
                label="maximum-likelihood estimate")
axes[0].set_xlabel("constant positive probability p")
axes[0].set_ylabel("negative log-likelihood")
axes[0].set_title("Observed labels determine the best constant probability")
axes[0].legend()
k_future_lab9 = np.arange(6)
future_masses_lab9 = np.array([binomial_pmf(int(k), 5, p_hat_lab9) for k in k_future_lab9])
axes[1].bar(k_future_lab9, future_masses_lab9)
axes[1].set_xticks(k_future_lab9)
axes[1].set_xlabel("positive labels in next five")
axes[1].set_ylabel("probability")
axes[1].set_title("Predictive Binomial distribution")
plt.tight_layout()
plt.show()


## Integrated lab tasks

1. Compute the likelihood and negative log-likelihood at $$p=0.5$$ and $$p=0.6.$$
2. Verify that the negative log-likelihood is smaller at $$p=0.6.$$
3. Compute the mean and variance of the next-five positive count.
4. Compute the probability of at most two positives.
5. Explain what assumptions are required to use the Binomial predictive distribution.


In [ ]:
reveal_markdown("Open integrated-lab solution", r"""
At $$p=0.5,$$

$$
\mathcal{L}(0.5)=(0.5)^3(0.5)^2=(0.5)^5=0.03125.
$$

The negative log-likelihood is

$$
-\ln(0.03125)\approx 3.465736.
$$

At $$p=0.6,$$

$$
\mathcal{L}(0.6)=(0.6)^3(0.4)^2
=(0.216)(0.16)
=0.03456.
$$

The negative log-likelihood is

$$
-\ln(0.03456)\approx 3.365058,
$$

which is smaller.

For $$X\sim\operatorname{Binomial}(5,0.6),$$

$$
E[X]=np=5(0.6)=\boxed{3},
$$

and

$$
\operatorname{Var}(X)=np(1-p)
=5(0.6)(0.4)
=\boxed{1.2}.
$$

The probability of at most two positives is

$$
P(X\leq 2)
=\sum_{k=0}^{2}\binom{5}{k}(0.6)^k(0.4)^{5-k}.
$$

Compute each term:

$$
P(X=0)=(0.4)^5=0.01024,
$$

$$
P(X=1)=5(0.6)(0.4)^4=0.0768,
$$

$$
P(X=2)=10(0.6)^2(0.4)^3=0.2304.
$$

Therefore

$$
P(X\leq 2)=0.01024+0.0768+0.2304
=\boxed{0.31744}.
$$

The predictive Binomial model assumes a fixed trial count, a common probability $$0.6,$$ independent trials, and the same data-generating behavior in the future.
""")


---

# Module 9 Summary

1. A random variable maps outcomes to numbers.
2. PMFs assign probabilities to discrete values; PDFs assign densities to continuous values.
3. CDFs accumulate probability up to thresholds.
4. Bernoulli models one binary trial, and Binomial models a success count across fixed independent trials.
5. Poisson models event counts using a rate parameter.
6. Exponential models nonnegative waiting times with constant-rate memorylessness.
7. Gaussian models symmetric continuous variation through mean and variance.
8. Initialization scale affects signal variance across a layer.
9. Binary cross-entropy is Bernoulli negative log-likelihood.

## Cumulative distribution guide

| Data mechanism | Candidate distribution |
|---|---|
| One yes-or-no outcome | Bernoulli |
| Success count in fixed independent trials | Binomial |
| Event count over a fixed interval | Poisson |
| Waiting time under a constant event rate | Exponential |
| Symmetric continuous noise or measurement | Gaussian |
| One class among several mutually exclusive labels | Categorical |

## Exit ticket

1. Why is a PDF height not a probability?
2. What assumptions separate a Binomial model from an arbitrary count model?
3. How are Poisson and Exponential distributions related conceptually?
4. Why should random weight variance depend on fan-in?
5. Why does cross-entropy heavily penalize confident incorrect predictions?
